In [9]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain.chat_models import init_chat_model
from langchain_core.tools import tool

@tool
def calculator(expression: str) -> str:
    """사칙연산 수식 문자열을 계산한다. 예: '12345 * 6789'"""
    try:
        result = eval(expression, {"__builtins__": {}})
        return str(result)
    except Exception as e:
        return f"계산 오류: {e}"

@tool
def word_count(text: str) -> str:
    """텍스트의 단어 수와 글자 수를 센다."""
    return f"단어 수: {len(text.split())}, 글자 수: {len(text)}"

MODEL = "google_genai:gemini-flash-latest"
model = init_chat_model(MODEL, temperature=0)
model_with_tools = model.bind_tools([calculator, word_count])

In [10]:
response = model_with_tools.invoke("12345 곱하기 6789는 얼마야?")
response

AIMessage(content=[], additional_kwargs={'function_call': {'name': 'calculator', 'arguments': '{"expression": "12345 * 6789"}'}, '__gemini_function_call_thought_signatures__': {'gMmTJI4R': 'EsABCr0BARFNMg8NyE4BEBlsdoRa+UJzaX+pl8uCpKnamJSrCQE6Q67pu0ngCk3/LOiKZHr0m+6HcFIRh5/7swSdb5+s9izqu1pbm5jpc8AlY4bEOI2/qBTWV2+FCqxND4oOmCrdSMlYpW0ueZ27fK56mnIufsytkIbKVgzSyt7esqThwwotU7LQtjxXOaocSFER5XaoF2RXuubBl7q/GfOG2iubEy72HmVhRxJf2IEgBNsm7kR3Uo8sCw2EzGYq4nk8'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.6-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019fea40-66af-7630-bd31-669a5c106bfb-0', tool_calls=[{'name': 'calculator', 'args': {'expression': '12345 * 6789'}, 'id': 'gMmTJI4R', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 129, 'output_tokens': 75, 'total_tokens': 204, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 51}})

In [12]:
for call in response.tool_calls:
    print("도구 이름:", call["name"])
    print("인자:", call["args"])
    if call["name"] == "calculator":
        print("실행 결과:", calculator.invoke(call["args"]))

도구 이름: calculator
인자: {'expression': '12345 * 6789'}
실행 결과: 83810205


In [13]:
for call in response.tool_calls:
    print("도구 이름:", call["name"])
    print("인자:", call["args"])
    if call["name"] == "calculator":
        print("실행 결과:", calculator.invoke(call["args"]))

도구 이름: calculator
인자: {'expression': '12345 * 6789'}
실행 결과: 83810205


In [14]:
from datetime import datetime

@tool
def get_current_time() -> str:
    """현재 시각을 반환한다."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

model_with_tools = model.bind_tools([calculator, word_count, get_current_time])

response2 = model_with_tools.invoke("지금 뉴욕은 몇 시야?")
response2.tool_calls

[{'name': 'get_current_time',
  'args': {},
  'id': '4CZWsn3P',
  'type': 'tool_call'}]

In [15]:
from zoneinfo import ZoneInfo

@tool
def get_current_time_with_timezone(timezone: str) -> str:
    """주어진 IANA 시간대(예: 'Asia/Seoul', 'America/New_York')의 현재 시각을 반환한다."""
    try:
        now = datetime.now(ZoneInfo(timezone))
        return now.strftime("%Y-%m-%d %H:%M:%S %Z")
    except Exception as e:
        return f"시간대 오류: {e}"

model_with_tools = model.bind_tools([calculator, word_count, get_current_time_with_timezone])

response3 = model_with_tools.invoke("서울이랑 뉴욕 지금 몇 시야?")
response3.tool_calls

[{'name': 'get_current_time_with_timezone',
  'args': {'timezone': 'Asia/Seoul'},
  'id': 'kyCflwS5',
  'type': 'tool_call'},
 {'name': 'get_current_time_with_timezone',
  'args': {'timezone': 'America/New_York'},
  'id': 'BaajfG3J',
  'type': 'tool_call'}]

In [18]:
from langchain_core.messages import HumanMessage, ToolMessage

messages3 = [HumanMessage(content="지금 서울, 마추피츄, 마드리드, 런던, 파리 지금 몇 시야?")]
ai_message3 = model_with_tools.invoke(messages3)
messages3.append(ai_message3)

for call in ai_message3.tool_calls:
    result = get_current_time_with_timezone.invoke(call["args"])
    messages3.append(ToolMessage(content=result, tool_call_id=call["id"]))

final_response = model_with_tools.invoke(messages3)
final_response.content

[{'type': 'text',
  'text': '요청하신 도시들의 현재 시간입니다:\n\n* **서울 (한국):** 2026년 8월 10일 오후 3시 7분\n* **마추피츄 (페루):** 2026년 8월 10일 오전 1시 7분\n* **마드리드 (스페인):** 2026년 8월 10일 오전 8시 7분\n* **런던 (영국):** 2026년 8월 10일 오전 7시 7분\n* **파리 (프랑스):** 2026년 8월 10일 오전 8시 7분',
  'extras': {'signature': 'EuoDCucDARFNMg8x7FP1uEpGnwqqkPz/033UvUjgM6N/Oktqh2ALSDyquwCQvnurkUvsJzx4W8SQJ3yBMYxHC3IpkgSgeGjaL9S52OQg5ssupEkuaSOi4QnM+8ROj7Jw+eK+cLMMcl2dkeZmol8zclTXw+zfgqJMR1DhXQGB/OqMEPKJKFmxJ0btbqaD/3RF0d4PDmtMMoLDyTveTAdeXaoIrG+IqA5gtpkgE0oCSirauv1jU1QKnftpRkmjNSlAkewdaSTRh4rbsHCwocnWaNUhlvDUA56a40Pss9nzl0ZVWHOEslvrxdlcGT5nlaE9144ILt7OgF33pa2bIfNuN7CPe2CtYwZfAqK7tQ+rqw2Kb2/GL6EVvxbMXHHYFtqSsSGCZ023g0hR32IVkFSIHS5RDq88EaB1GgG562Pd5KBsGvCFoGqF8RYi08Mwmu0ALoqnMOElB/0h1pCUHFItCyTfK433v+fJyIjgKpdnbLOIURzak3OXTzM/TWmjDWJbknVZtrmdxz11ZgqRr3BVVsd6NaaroToOhjoNa0f2ov9GMX8TBL5PSQxZ2yn9KDtT8vsXHNH1iaRZ8ofubdhCjAZwJRsW6xpefWHfAk1X0GDrFPYRRC22nToj6xY6kN94uSo3z4nPm1xBqq49rQ=='}}]

In [20]:
print("도구 호출 개수:", len(ai_message3.tool_calls))
for call in ai_message3.tool_calls:
    print("-", call["args"])

도구 호출 개수: 5
- {'timezone': 'Asia/Seoul'}
- {'timezone': 'America/Lima'}
- {'timezone': 'Europe/Madrid'}
- {'timezone': 'Europe/London'}
- {'timezone': 'Europe/Paris'}
